In [54]:
from zipper_owntry import Zipp3r

In [55]:
# image_in = '/app/data/data3channels-002.tif'
image_in = '/app/data/data1channel-001.tif'
output_path = '/app/Zipping-pruebas/OUTPUT'

In [56]:
zipper = Zipp3r(image_in, output_path)

# Create a dictionary containing information on all blocks
zipper.set_block_slices(
    image_shape=[146, 3788, 3788],
    blocksize=[146, 1894, 1894],
    margins=[0,100,100]
)

In [57]:
# Cut the image according to the block_slices
zipper.split_image()

array([[[ 335,    6,   13, ...,    5,   10,   34],
        [   9,   13,   15, ...,    0,   15,   14],
        [   0,   12,   16, ...,    0,    0,    3],
        ...,
        [  13,    5,   21, ...,    9,   20,   22],
        [   8,    2,   16, ...,  901,    9,   21],
        [  16,    4,   18, ...,    9,   14,  339]],

       [[   2,   12,    4, ...,   22,  946,    2],
        [   2,   13,   13, ...,   15,   22,   15],
        [   0,   10,    7, ...,    0,    2,   21],
        ...,
        [  14,   13,    0, ...,   12,    8,    9],
        [  19,    3,    6, ...,   11,   27,   26],
        [   6,    6,  150, ...,   25,   15,   19]],

       [[   4,    0,    3, ...,    3,   14,    0],
        [   2,  453,   21, ...,   23,   19,    8],
        [   0,    4,  683, ...,    9,    7,    6],
        ...,
        [  11,    8,   15, ...,   23,   14,   12],
        [  15,   13,   26, ...,   14,    5,   15],
        [  16,   10,    0, ...,    9,    8,    7]],

       ...,

       [[ 311,  164,   1

In [58]:
import os
import numpy as np
from PIL import Image
import tifffile

def reconstruct_image(tiles_folder, output_filename, tile_size, overlap):
    # List all TIFF files in the tiles_folder
    tiles = [f for f in os.listdir(tiles_folder) if f.endswith('.tif')]
    
    if not tiles:
        raise FileNotFoundError("No TIFF files found in the specified folder.")
    
    # Sort the tiles based on their Z, X, and Y indices
    def extract_indices(filename):
        match = re.search(r'_block(\d+)x(\d+)x(\d+)\.tif$', filename)
        if match:
            x, y, z = map(int, match.groups())
            return (z, x, y)
        else:
            return (float('inf'), float('inf'), float('inf'))

    tiles.sort(key=extract_indices)
    
    # Open the first tile to get the dimensions of each stack
    first_tile_path = os.path.join(tiles_folder, tiles[0])
    with tifffile.TiffFile(first_tile_path) as tif:
        num_slices = len(tif.pages)
        tile_sample = tif.pages[0].asarray()
        tile_height, tile_width = tile_sample.shape
    
    # Determine the number of tiles in each dimension
    num_tiles_x = num_tiles_y = max(x for _, x, _ in (extract_indices(tile) for tile in tiles)) + 1
    num_tiles_z = max(z for z, _, _ in (extract_indices(tile) for tile in tiles)) + 1
    
    # Calculate the size of the final image
    full_width = num_tiles_x * (tile_size[1] - overlap) + tile_size[1]
    full_height = num_tiles_y * (tile_size[0] - overlap) + tile_size[0]
    full_depth = num_slices
    
    # Create an empty array to hold the full image stack
    full_image_stack = np.zeros((full_depth, full_height, full_width), dtype=np.uint16)
    
    # Load and place each tile into the full image stack
    for tile_name in tiles:
        # Extract indices from the tile name
        z, x, y = extract_indices(tile_name)
        
        # Calculate the position of the tile in the full image
        x_start = x * (tile_size[1] - overlap)
        y_start = y * (tile_size[0] - overlap)
        
        # Open the tile image stack
        tile_path = os.path.join(tiles_folder, tile_name)
        with tifffile.TiffFile(tile_path) as tif:
            for slice_idx, page in enumerate(tif.pages):
                if slice_idx < full_depth:
                    tile_array = page.asarray()
                    full_image_stack[slice_idx, y_start:y_start + tile_size[0], x_start:x_start + tile_size[1]] = tile_array
    
    # Save the final image stack as a TIFF file
    tifffile.imsave(output_filename, full_image_stack)

# Define your variables
tiles_folder = '/app/Zipping-pruebas/OUTPUT/blocks'
output_filename = '/app/Zipping-pruebas/OUTPUT/final.tif'
tile_size = (1994, 1994)  # Adjust according to your actual tile size
overlap = 100  # Adjust according to your overlapping size

# Call the function
reconstruct_image(tiles_folder, output_filename, tile_size, overlap)

/tmp/ipykernel_3668/4256274028.py:61: DeprecationWarning: <tifffile.imsave> is deprecated. Use tifffile.imwrite
  tifffile.imsave(output_filename, full_image_stack)
